# 변경사항
## 0625 버전
- 불성실 응답 기준 재정의
  - 설문 불성실 판단 기준
    - 설문내 두 PSF 모두 동일 응답 비율이 95% 이상인 경우
    - ex) PSF 1, 2 모두 보통 응답 or PSF 1은 모두 보통, PSF 2는 모두 나쁨 응답
  - 회생 케이스
    - 다만 같은 연도내 응답수가 30이상이며 성실하게 수행한 설문이 있는 경우 해당 연도의 불성실 응답은 성실히 수행한 것으로 처리
## 0622 버전
- 불성실 응답자 경우 추가
  - 기존엔 설문단일응답을 기준으로 불성실을 골라냈으나, psf1은 나쁨, psf2는 좋음으로 쭉 찍은 경우도 존재
  - 이 경우도 불성실로 처리

In [ ]:
import pandas as pd
import json
import numpy as np
from pathlib import Path
import shutil

pd.set_option('display.max_rows', None)      # 모든 행 출력
pd.set_option('display.max_columns', None)   # 모든 열 출력
pd.set_option('display.width', None)         # 출력 너비 제한 해제
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 길이 제한 해제

# 1. 데이터 특성 확인 및 전처리
## 1.1 job, answer, health 테이블 분리
### 1.1.1 questions 컬럼 분해

In [ ]:
df = pd.read_csv("data.csv")
df.head()
df.columns

In [ ]:
# 원래 id는 data_id로 이름 변경
df = df.rename(columns={
    "id": "data_id",
    "round": "set_id"
})

# data 컬럼이 JSON 문자열이면 파싱
df["data_parsed"] = df["questions"].apply(json.loads)

# data 안의 리스트를 행으로 분해
df_long = df.explode("data_parsed").reset_index(drop=True)

# JSON 내부 id는 question_id로 분리
df_long["answer"] = df_long["data_parsed"].apply(lambda x: x["answer"])
df_long["PSF"] = df_long["data_parsed"].apply(lambda x: x.get("number")).str.rstrip(".").astype(int)
df_long["conditions"] = df_long["data_parsed"].apply(lambda x: x.get("conditions", {}))
df_long = df_long.drop(columns=["data_parsed", "questions"])
print("df shape", df.shape, "df_long shape:", df_long.shape)


In [ ]:
# 테스트 데이터 제거
df_long = df_long[df_long["name"] != "홍길동"]

In [ ]:
df_long[(df_long["PSF"]==1)]["answer"].value_counts()

### 1.1.2 사람 기준 고유 id 생성

In [ ]:
# 고유값으로 사용할 인적 데이터 확인
print("이름:", df["name"].nunique())
print("employee_id:", df["employee_id"].nunique())
print("employ_card_number:", df["employee_card_number"].nunique())

In [ ]:
# 이름이 같은데 다른 사번을 가진 인원 추출
name_to_empid = (
    df.groupby("name")["employee_id"]
    .nunique()
    .reset_index(name="empid_count")
)

names = name_to_empid.loc[
    name_to_empid["empid_count"] > 1,
    "name"
]
names

In [ ]:
# name, birth를 통해 동명이인 여부 확인
df_long.loc[
    df_long["name"].isin(names),
    ["name", "employee_id", "date_of_birth",]
].drop_duplicates().sort_values(["name", "employee_id"])

In [ ]:
# 아래 리스트는 사번 잘못 기입으로 보고 수정
empid_error_list = ["김봉규", "박찬홍", "이승식", "최진호", "최한검"]

df_long["employee_id_orig"] = df_long["employee_id"]

mask = df_long["name"].isin(empid_error_list)

# (name, birth)별 대표 employee_id
empid_map = (
    df_long.loc[mask]
    .groupby(["name", "date_of_birth"])["employee_id"]
    .agg(lambda x: x.mode().iloc[0])
)

# 적용
df_long.loc[mask, "employee_id"] = (
    df_long.loc[mask]
    .set_index(["name", "date_of_birth"])
    .index.map(empid_map)
)

In [ ]:
# name, birth를 통해 동명이인 재확인
df_long.loc[
    df_long["name"].isin(names),
    ["name", "employee_id", "date_of_birth",]
].drop_duplicates().sort_values(["name", "employee_id"])

In [ ]:
# 김민길씨는 같은 사람으로 보고 사번을 830341로 변경
df_long.loc[
    (df_long["name"] == "김민길") &
    (df_long["date_of_birth"] == "1965-02-10"),
    "employee_id"
] = "830341"

In [ ]:
# 이름이 같은데 다른 사번을 가진 인원 추출
empid_to_name = (
    df.groupby("employee_id")["name"]
    .nunique()
    .reset_index(name="name_count")
)

empids = empid_to_name.loc[
    empid_to_name["name_count"] > 1,
    "employee_id"
]
empids

In [ ]:
# 다른 사람으로 판단되어 그대로 유지
df_long.loc[
    df_long["employee_id"].isin(empids),
    ["name", "employee_id", "date_of_birth",]
].drop_duplicates().sort_values(["name", "employee_id"])

In [ ]:
df_long["empid"] = df_long["name"] + "_" + df_long["employee_id"]

### 1.1.3 테이블 분해 및 저장
- person_job, person_health, answer 테이블로 분해

In [ ]:
common_col = ["empid", "saved_at"]

In [ ]:
# 직업 테이블 분해
df_job_col = common_col + ["name",  'affiliation', 'job', 'years', 'employee_id', 'position',
       'department', 'railroad_certification', 'job_education', 'work_type', 'work_time','occupation_detail', 'overtime_period', 'overtime_long']

def longest_value(x):
    x = x.dropna()
    if len(x) == 0:
        return np.nan

    return max(x, key=lambda v: len(str(v)))

df_job = (
    df_long[df_job_col]
    .groupby("empid", as_index=False)
    .agg(longest_value)
)

In [ ]:
df_job["empid"].nunique()

In [ ]:
# 건강정보 테이블 분해
df_health_col = common_col + ['name', 'gender', 'date_of_birth', 'health_check_date', 'body_temperature', 'systolic_bp', 'diastolic_bp', 'pulse']

df_health = df_long[df_health_col].drop_duplicates(subset=["empid"])

In [ ]:
df_health["empid"].nunique()

In [ ]:
# answer 테이블 분해
df_answer_col = common_col + ['set_id', "answer", "PSF", "conditions"]

df_answer = df_long[df_answer_col]

In [ ]:
df_answer.shape

In [ ]:
# 데이터 저장
from datetime import datetime

Path("processed_data/history").mkdir(parents=True, exist_ok=True)

for file in Path("processed_data").glob("*.csv"):
    shutil.move(
        str(file),
        str(Path("processed_data/history") / file.name)
    )

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

df_job.to_csv(f"processed_data/job_{timestamp}.csv", index=False, encoding="utf-8-sig")
df_health.to_csv(f"processed_data/health_{timestamp}.csv", index=False, encoding="utf-8-sig")
df_answer.to_csv(f"processed_data/answer_{timestamp}.csv", index=False, encoding="utf-8-sig")

## 1.2 answer 테이블 추가 전처리 수행
- 설문조사 id, 수행년도, 응답자수 확인

In [ ]:
import pandas as pd
import ast
import numpy as np
from pathlib import Path

pd.set_option('display.max_rows', None)      # 모든 행 출력
pd.set_option('display.max_columns', None)   # 모든 열 출력
pd.set_option('display.width', None)         # 출력 너비 제한 해제
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 길이 제한 해제

In [ ]:
latest_file = max(
    Path("processed_data").glob("answer_*.csv"),
    key=lambda x: x.stat().st_mtime
)

answer_df = pd.read_csv(latest_file)
answer_df.head(1)

### 1.2.1 설문조사 고유 id 생성

In [ ]:
# 설문조사 수행년도 생성
answer_df["saved_at"] = pd.to_datetime(answer_df["saved_at"], format="mixed",utc=True)
answer_df["survey_year"] = answer_df["saved_at"].dt.year

# empid, 저장시간 바탕으로 설문조사 id 생성

answer_df["set_id"] = answer_df["set_id"].astype(int)
answer_df = answer_df.sort_values(["empid", "saved_at", "set_id"]).copy()

answer_df["prev_set_id"] = answer_df.groupby("empid")["set_id"].shift()

answer_df["new_survey"] = (
    answer_df["prev_set_id"].isna() |
    (answer_df["set_id"] < answer_df["prev_set_id"])
)

answer_df["survey_no"] = answer_df.groupby("empid")["new_survey"].cumsum()

answer_df["survey_id"] = (
    answer_df["empid"] + "_" +
    answer_df["survey_year"].astype(str) + "_" +
    answer_df["survey_no"].astype(str)
)
answer_df = answer_df.drop(columns=["new_survey", "prev_set_id"])

In [ ]:
answer_df.head(2)

### 1.2.2 psf를 숫자에서 문자로 변환

In [ ]:
# psf와 conditions 잘 매핑되었는지 확인
answer_df["conditions"] = answer_df["conditions"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# 2. conditions 내부 key 목록 추출
answer_df["condition_keys"] = answer_df["conditions"].apply(
    lambda x: tuple(sorted(x.keys())) if isinstance(x, dict) else None
)

psf_key_check = (
    answer_df
    .groupby("PSF")["condition_keys"]
    .nunique()
    .reset_index(name="key_pattern_count")
)

psf_key_check

In [ ]:
# psf id별 키 목록
psf_key_map = (
    answer_df
    .groupby("PSF")["condition_keys"]
    .first()
    .reset_index()
)

psf_key_map

In [ ]:
psf_map = {
    1: "작업부하",
    2: "장비 및 기기",
    3: "절차 및 지침",
    4: "교육 및 훈련",
    5: "의사소통",
    6: "환경",
    7: "작업절차",
    8: "복잡성"
}

answer_df["PSF_name"] = answer_df["PSF"].map(psf_map)

# survey_id + set_id 안에서 같은 PSF가 반복되는 예외 케이스를 안전하게 구분
answer_df["repeat_no"] = (
    answer_df
    .groupby(["survey_id", "set_id", "PSF_name"])
    .cumcount()
)


In [ ]:
answer_df.head(2)

### 1.2.3 answer 응답자 수 확인

#### 1.2.3.1 년도별 고유 응답자

In [ ]:
# 1차년도 설문조사 수행 고유 응답자
emp_2025_id = set(answer_df.loc[answer_df["survey_year"] == 2025, "empid"])
answer_df[answer_df["survey_year"]==2025]["empid"].nunique()

In [ ]:
# 2차년도 설문조사 수행 고유 응답자
emp_2026_id = set(answer_df.loc[answer_df["survey_year"] == 2026, "empid"])
answer_df[answer_df["survey_year"]==2026]["empid"].nunique()

In [ ]:
# 1, 2차 설문 참여자
both_years = emp_2025_id & emp_2026_id
len(both_years)

In [ ]:
# 전체 고유 응답자
answer_df["empid"].nunique()

#### 1.2.2.2 년도별 설문 수행 건수

In [ ]:
emp_2025 = answer_df[answer_df["survey_year"]==2025]
emp_2026 = answer_df[answer_df["survey_year"]==2026]

In [ ]:
# 총 수행 문항수
print("2025년:",len(emp_2025),", 2026년:",len(emp_2026), "총합: ",len(answer_df))

In [ ]:
emp_2025["PSF_name"].value_counts()

In [ ]:
emp_2026["PSF_name"].value_counts()

In [ ]:
answer_df["PSF_name"].value_counts()

### 1.2.4 불성실 응답자 라벨링
- 아래 케이스에서 95%초과 단일 응답을 한 경우 불성실로 간주, 90%초과는 의심 케이스로 분리
- 불성실 의심 CASE와 처리 방안은 다음과 같음
1. 1개 이상 설문을 했고, 모든 설문을 단일 응답한 비율>=95% -> 모두 불성실로 처리
2. 2개년도 중 특정 연도만 단일 응답한 비율>=95% -> 해당 연도 설문만 불성실로 처리
3. 같은 연도에 다른 설문을 모두 단일응답하다가 본인 설문을 성실히 수행 -> 두 케이스 모두 성실로 처리
4. 설문 내에서 PSF별로 각각 단일응답한 비율>=95% (PSF 1은 나쁨, PSF 2는 모두 좋음을 선택) -> 불성실로 처리

In [ ]:
YEAR_NORMAL_N_ANSWER = 30

QUALITY_RULES = {
    "quality_100": 1.00,
    "quality_95": 0.95,
    "quality_90": 0.90,
}

In [ ]:
# PSF별 응답 분포 계산
psf_answer_dist = (
    answer_df
    .groupby(["empid", "survey_year", "survey_id", "PSF_name"])["answer"]
    .value_counts()
    .reset_index(name="answer_count")
)

psf_quality = (
    psf_answer_dist
    .groupby(["empid", "survey_year", "survey_id", "PSF_name"])
    .agg(
        psf_n_answer=("answer_count", "sum"),
        psf_max_answer_count=("answer_count", "max")
    )
    .reset_index()
)

psf_quality["psf_max_answer_ratio"] = (
    psf_quality["psf_max_answer_count"]
    / psf_quality["psf_n_answer"]
)

In [ ]:
# PSF 별 기준 충족 여부 확인
for quality_col, ratio in QUALITY_RULES.items():
    psf_quality[f"{quality_col}_psf_flag"] = (
        psf_quality["psf_max_answer_ratio"] >= ratio
    )

In [ ]:
# 설문 단위 품질 테이블 생성
# 설문 내 두 PSF 모두 동일 응답 비율이 기준 이상이면 불성실
survey_quality_new = (
    psf_quality
    .groupby(["empid", "survey_year", "survey_id"])
    .agg(
        n_psf=("PSF_name", "nunique"),
        n_answer=("psf_n_answer", "sum"),
        quality_100_psf_count=("quality_100_psf_flag", "sum"),
        quality_95_psf_count=("quality_95_psf_flag", "sum"),
        quality_90_psf_count=("quality_90_psf_flag", "sum"),
    )
    .reset_index()
)

In [ ]:
for quality_col in QUALITY_RULES.keys():
    count_col = f"{quality_col}_psf_count"
    raw_col = f"{quality_col}_raw"

    survey_quality_new[raw_col] = "성실"

    survey_quality_new.loc[
        (survey_quality_new["n_psf"] == 2) &
        (survey_quality_new[count_col] == 2),
        raw_col
    ] = "불성실"

In [ ]:
# 같은 연도 내 성실 설문 예외 처리
# 같은 empid, 같은 survey_year 안에 응답수 30 이상 raw 기준 성실 설문이 하나라도 있으면 해당 연도 전체 불성실을 성실로 처리
year_exception = survey_quality_new[["empid", "survey_year"]].drop_duplicates()

for quality_col in QUALITY_RULES.keys():
    raw_col = f"{quality_col}_raw"
    exception_col = f"has_sincere_{quality_col.replace('quality_', '')}"

    tmp = (
        survey_quality_new
        .assign(
            sincere_over_30=lambda x:
                (x[raw_col] == "성실") &
                (x["n_answer"] >= YEAR_NORMAL_N_ANSWER)
        )
        .groupby(["empid", "survey_year"])
        .agg(
            **{exception_col: ("sincere_over_30", "any")}
        )
        .reset_index()
    )

    year_exception = year_exception.merge(
        tmp,
        on=["empid", "survey_year"],
        how="left"
    )

survey_quality_new = survey_quality_new.merge(
    year_exception,
    on=["empid", "survey_year"],
    how="left"
)

for quality_col in QUALITY_RULES.keys():
    raw_col = f"{quality_col}_raw"
    exception_col = f"has_sincere_{quality_col.replace('quality_', '')}"

    survey_quality_new[quality_col] = survey_quality_new[raw_col]

    survey_quality_new.loc[
        survey_quality_new[exception_col] == True,
        quality_col
    ] = "성실"

In [ ]:
quality_cols_to_merge = [
    "empid",
    "survey_year",
    "survey_id",
    "quality_100",
    "quality_95",
    "quality_90",
]

answer_df_new_quality = answer_df.merge(
    survey_quality_new[quality_cols_to_merge],
    on=["empid", "survey_year", "survey_id"],
    how="left"
)

### 1.2.5 응답 성실도 확인

In [ ]:
# 설문 단일응답자 비율 (True가 설문 단일 응답자)
survey_answer_check = (
    answer_df_new_quality
    .groupby(["empid", "survey_year", "survey_id"])
    .agg(
        n_answer=("answer", "size"),
        unique_answers=("answer", "nunique")
    )
    .reset_index()
)

survey_answer_check["uniform_survey"] = (
    survey_answer_check["unique_answers"] == 1
)

# 설문 전체가 완전 단일응답인 비율
survey_answer_check["uniform_survey"].value_counts(normalize=True) * 100

In [ ]:
# PSF 단일응답자 비율 (True가 PSF 단일 응답자)
psf_uniform_check = psf_quality.copy()

psf_uniform_check["uniform_psf_100"] = (
    psf_uniform_check["psf_max_answer_ratio"] >= 1.00
)

psf_uniform_check["uniform_psf_95"] = (
    psf_uniform_check["psf_max_answer_ratio"] >= 0.95
)

psf_uniform_check["uniform_psf_90"] = (
    psf_uniform_check["psf_max_answer_ratio"] >= 0.90
)

print("PSF 100% 동일응답 비율")
print(psf_uniform_check["uniform_psf_100"].value_counts(normalize=True) * 100)

print("\nPSF 95% 이상 동일응답 비율")
print(psf_uniform_check["uniform_psf_95"].value_counts(normalize=True) * 100)

print("\nPSF 90% 이상 동일응답 비율")
print(psf_uniform_check["uniform_psf_90"].value_counts(normalize=True) * 100)

In [ ]:
print("quality_100")
print(survey_quality_new["quality_100"].value_counts())

print("\nquality_95")
print(survey_quality_new["quality_95"].value_counts())

print("\nquality_90")
print(survey_quality_new["quality_90"].value_counts())

In [ ]:
# =========================================================
# 4. 24개 문항을 단일응답했지만 최종 성실 처리된 설문 확인
# =========================================================
target_survey_check = survey_answer_check.merge(
    survey_quality_new[
        [
            "empid",
            "survey_year",
            "survey_id",
            "quality_100",
            "quality_95",
            "quality_90"
        ]
    ],
    on=["empid", "survey_year", "survey_id"],
    how="left"
)

target_survey_check[
    (target_survey_check["n_answer"] == 24)
    & (target_survey_check["uniform_survey"] == True)
    & (
        (target_survey_check["quality_100"] == "성실")
        | (target_survey_check["quality_95"] == "성실")
        | (target_survey_check["quality_90"] == "성실")
    )
]

In [ ]:
# 해당 empid의 전체 설문 이력 확인
target_ids = target_survey_check.loc[
    (target_survey_check["n_answer"] == 24)
    & (target_survey_check["uniform_survey"] == True)
    & (
        (target_survey_check["quality_100"] == "성실")
        | (target_survey_check["quality_95"] == "성실")
        | (target_survey_check["quality_90"] == "성실")
    ),
    "empid"
].unique()

survey_quality_new[
    survey_quality_new["empid"].isin(target_ids)
].sort_values(["empid", "survey_year", "survey_id"])

In [ ]:
output_file = latest_file.with_name(
    latest_file.stem + "_new_quality" + latest_file.suffix
)

answer_df_new_quality.to_csv(output_file, index=False, encoding="utf-8-sig")

print("저장:", output_file)

## 1.3 job 테이블 추가 전처리 수행

In [ ]:
job_latest_file = max(
    Path("processed_data").glob("job_*.csv"),
    key=lambda x: x.stat().st_mtime
)

job_df = pd.read_csv(job_latest_file)

In [ ]:
job_df.head(2)

### 1.3.1 직장 확인

In [ ]:
# 직장은 모두 같으므로 한국철도공사로 통일
job_df["affiliation"] = "한국철도공사"
job_df["affiliation"].value_counts()

In [ ]:
# 승무원은 없는데 이건 어떻게 처리하는지?
job_df["job"].value_counts()

### 1.3.2 position 전처리 및 분리 수행

In [ ]:
job_df["position"].unique()

In [ ]:
# position 직군, 급수 추출
job_df["position"] = (
    job_df["position"]
    .astype(str)
    .str.strip()
    .str.replace(" ", "", regex=False)
)

# 오타 및 직군 통일
job_df["position"] = (
    job_df["position"]
    .str.replace("사영", "사무영업", regex=False)
    .str.replace("서무영업", "사무영업", regex=False)
    .str.replace("전통", "전기통신", regex=False)
    .str.replace("통목", "토목", regex=False)
    .str.replace("운잔", "운전", regex=False)
    .str.replace(r"^사무(?=\d|급)", "사무영업", regex=True)
)
job_df["position"].unique()

In [ ]:
job_df["job_family"] = job_df["position"].str.extract(r"([가-힣]+)")

In [ ]:
job_df["grade"] = pd.to_numeric(
    job_df["position"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

In [ ]:
job_df["job_family"].unique()

### 1.3.3 work_type 정리

In [ ]:
job_df["work_type"].unique()

In [ ]:
job_df["work_type_clean"] = job_df["work_type"].astype(str)
job_df["work_type_clean"] = (
    job_df["work_type_clean"]
    .str.replace(" ", "", regex=False)
    .str.strip()
)

In [ ]:
job_df.loc[
    job_df["work_type_clean"].str.contains("교번|교변", na=False),
    "work_type_clean"
] = "교번"

job_df.loc[
    job_df["work_type_clean"].str.contains("일근", na=False),
    "work_type_clean"
] = "일근"

job_df.loc[
    job_df["work_type_clean"].str.contains("3조.*2교대|3조2교대", na=False),
    "work_type_clean"
] = "3조2교대"

job_df.loc[
    job_df["work_type_clean"].str.contains("4조.*2교대|4조2교대|4조2교재", na=False),
    "work_type_clean"
] = "4조2교대"

job_df.loc[
    job_df["work_type_clean"].str.contains("시차", na=False),
    "work_type_clean"
] = "시차근무"

In [ ]:
job_df["work_type_clean"].value_counts(dropna=False)

In [ ]:
job_df.head(2)

In [ ]:
# 가져갈 컬럼 선택
job_df_col = ["empid", "name", "affiliation", "job", "years", "department", "railroad_certification", "job_education",
              "work_time", "job_family", "grade", "work_type_clean"]
job_df = job_df[job_df_col]

In [ ]:
output_file = job_latest_file.with_name(
    job_latest_file.stem + job_latest_file.suffix
)

job_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("저장:", output_file)

## 1.4 health 테이블 추가 전처리 수행

In [ ]:
health_latest_file = max(
    Path("processed_data").glob("health_*.csv"),
    key=lambda x: x.stat().st_mtime
)

health_df = pd.read_csv(health_latest_file)

In [ ]:
health_df.head(2)

### 1.4.1 연령대 그룹 추가

In [ ]:
health_df["date_of_birth"] = pd.to_datetime(health_df["date_of_birth"], format="mixed",utc=True)
health_df["age"] = 2026 - health_df["date_of_birth"].dt.year
health_df["age_group"] = pd.cut(
    health_df["age"],
    bins=[18, 29, 39, 49, 59, 150],
    labels=["20대 이하", "30대", "40대", "50대", "60대 이상"]
)
health_df["age_group"] = (
    health_df["age_group"]
    .cat.add_categories(["미기입"])
    .fillna("미기입")
)

In [ ]:
health_df["age_group"].value_counts()

In [ ]:
health_df.head()

### 1.4.2 심박수 전처리

In [ ]:
health_df[health_df["systolic_bp"]<=100]

In [ ]:
# 90보다 작은 사람들은 반대로 적힌것이므로 변환
mask = health_df["systolic_bp"] <= 90

health_df.loc[mask, ["systolic_bp", "diastolic_bp"]] = (
    health_df.loc[mask, ["diastolic_bp", "systolic_bp"]].values
)

In [ ]:
health_df.head(2)

In [ ]:
output_file = health_latest_file.with_name(
    health_latest_file.stem + health_latest_file.suffix
)

health_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("저장:", output_file)

# 2. Answer 테이블 questions 분해
- 시트의 내용과 answer의 키, 값들이 살짝씩 다르기에 수기로 매핑 수행

In [ ]:
import pandas as pd
import ast
import json
import numpy as np
from pathlib import Path

pd.set_option('display.max_rows', None)      # 모든 행 출력
pd.set_option('display.max_columns', None)   # 모든 열 출력
pd.set_option('display.width', None)         # 출력 너비 제한 해제
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 길이 제한 해제

## 2.1 mapping 데이터 생성

In [ ]:
answer_latest_file = max(
    Path("processed_data").glob("answer_*_new_quality.csv"),
    key=lambda x: x.stat().st_mtime
)

answer_df = pd.read_csv(answer_latest_file)
answer_df.head(1)

In [ ]:
# answer_df["conditions"] = answer_df["conditions"].apply(
#     lambda x: ast.literal_eval(x) if isinstance(x, str) else x
# )

In [ ]:
# # dict의 key-value를 리스트로 변환
# cond_long = answer_df.copy()

# # dict를 key-value 리스트로 변환
# cond_long["condition_items"] = cond_long["conditions"].apply(
#     lambda x: list(x.items()) if isinstance(x, dict) else []
# )

# # 행 분리
# cond_long = cond_long.explode("condition_items").reset_index(drop=True)

# # Sub-PSF, Sub-PSF LV 추출
# cond_long["Sub-PSF"] = cond_long["condition_items"].str[0]
# cond_long["Sub-PSF LV."] = cond_long["condition_items"].str[1]

# cond_long = cond_long.drop(
#     columns=["condition_items"]
# )
# cond_long.head(1)

In [ ]:
# subpsf_values = (
#     cond_long[
#         ["PSF_name", "Sub-PSF", "Sub-PSF LV."]
#     ]
#     .drop_duplicates()
#     .sort_values(
#         ["PSF_name", "Sub-PSF", "Sub-PSF LV."]
#     )
#     .reset_index(drop=True)
# )

In [ ]:
# # 실제 conditions에서 나온 고유 조합 기준 매핑표 생성
# mapping_df = (
#     subpsf_values
#     .copy()
#     .sort_values(["PSF_name", "Sub-PSF", "Sub-PSF LV."])
#     .reset_index(drop=True)
# )

# # 임시 level 자동 부여
# mapping_df["level_auto"] = (
#     mapping_df
#     .groupby(["PSF_name", "Sub-PSF"])
#     .cumcount() + 1
# )

# # 사람이 수정할 최종 level 컬럼
# mapping_df["level"] = mapping_df["level_auto"]

# mapping_df.to_excel(
#     "subpsf_level_mapping_review.xlsx",
#     index=False
# )

In [ ]:
# def clean_text(x):
#     if pd.isna(x):
#         return x

#     return (
#         str(x)
#         .replace("\xa0", " ")
#         .replace("  ", " ")
#         .strip()
#     )


# # 2. 원하는 PSF / Sub-PSF 순서 정의
# psf_subpsf_order = {
#     "작업부하": [
#         "근무시간",
#         "작업시간",
#         "수면시간",
#         "작업 허용시간",
#         "근무시간대",
#     ],
#     "장비 및 기기": [
#         "장비 가용성(접근성)",
#         "작업 도구의 적절성",
#         "컴퓨터 지원 시스템의 적절성",
#         "장비의 사용성",
#         "장비 사용매뉴얼의 질",
#         "타 기기와의 연계성",
#         "유지보수의 적절성",
#     ],
#     "절차 및 지침": [
#         "(직무)절차의 유무(가용성)",
#         "(직무)절차 기술(description)의 복잡성",
#         "(직무)절차의 명료성(구체성)",
#         "(직무)절차의 적절성",
#         "(직무)절차의 일관성",
#         "(직무)절차의 추적성",
#     ],
#     "교육 및 훈련": [
#         "교육 프로그램 적절성",
#         "교육 주기(빈도)",
#         "상황에 대한 경험정도(자주 or 드물게 겪는지)",
#         "직무능력 평가 빈도",
#         "시스템 지식(원리적 지식)의 숙련도",
#         "절차적 지식의 숙련도",
#         "장비 사용 지식의 숙련도",
#     ],
#     "의사소통": [
#         "의사소통 메시지의 길이",
#         "의사소통을 위한 장비/ 기기의 적절성",
#         "의사소통을 위한 전문용어 숙지 필요성",
#         "의사소통 프로토콜의 구체성",
#         "의사소통자간 상황인식 공유 정도",
#         "팀/조직 의사소통 자율성",
#     ],
#     "환경": [
#         "풍속",
#         "비",
#         "실외습도",
#         "실외온도",
#         "낙뢰",
#         "안개",
#         "황사",
#         "작업 공간",
#         "마찰력",
#         "실내온도",
#         "실내습도",
#         "통풍",
#         "소음",
#         "업무공간의 가시성",
#     ],
#     "작업절차": [
#         "작업 관리 및 감독 적절성",
#         "작업 계획 적절성",
#         "작업절차 및 규정",
#         "역할 분담 명료화",
#         "작업자간 협업 정도",
#     ],
#     "복잡성": [
#         "의사결정 목표의 명확성",
#         "의사결정 대안의 수",
#         "의사결정 판단 기준의 수",
#         "(동시에 처리 되어야 하는)의사결정 수",
#         "관측되는 정보의 양",
#         "(관측이 불가능하여) 추론해야 하는 정보의 양",
#         "정보의 불확실성",
#     ],
# }


# # 3. 순서표 DataFrame 생성
# order_rows = []

# for psf_order, (psf, subpsfs) in enumerate(psf_subpsf_order.items(), start=1):
#     for subpsf_order, subpsf in enumerate(subpsfs, start=1):
#         order_rows.append({
#             "PSF_name_clean": clean_text(psf),
#             "Sub-PSF_clean": clean_text(subpsf),
#             "psf_order": psf_order,
#             "subpsf_order": subpsf_order,
#         })

# order_df = pd.DataFrame(order_rows)

# # 4. mapping_df의 표기 차이 보정
# manual_subpsf_fix = {
#     ("교육 및 훈련", "교육/훈련 프로그램 적절성"): "교육 프로그램 적절성",
#     ("교육 및 훈련", "상황에 대한 경험 정도(자주 또는 드물게 겪는지)"): "상황에 대한 경험정도(자주 or 드물게 겪는지)",
#     ("복잡성", "(관측이 불가능하여)추론해야하는 정보의 양"): "(관측이 불가능하여) 추론해야 하는 정보의 양",
#     ("의사소통", "의사소통을 위한 장비/기기의 적절성"): "의사소통을 위한 장비/ 기기의 적절성",
#     ("의사소통", "의사소통자 간 상황인식 공유 정도"): "의사소통자간 상황인식 공유 정도",
#     ("작업부하", "작업 허용 시간"): "작업 허용시간",
#     ("장비 및 기기", "장비 사용 매뉴얼의 질"): "장비 사용매뉴얼의 질",
#     ("절차 및 지침", "(직무)절차의 적절성(완전성 포함)"): "(직무)절차의 적절성",
#     ("환경", "마찰계수"): "마찰력",
#     ("환경", "실내 습도"): "실내습도",
#     ("환경", "실내 온도"): "실내온도",
#     ("환경", "실외 습도"): "실외습도",
#     ("환경", "실외 온도"): "실외온도",
#     ("환경", "업무공간 가시성"): "업무공간의 가시성",
#     ("환경", "작업공간 크기"): "작업 공간",
# }

# def fix_subpsf_name(row):
#     key = (row["PSF_name"], row["Sub-PSF"])
#     return manual_subpsf_fix.get(key, row["Sub-PSF"])


# mapping_df["Sub-PSF_fixed"] = mapping_df.apply(fix_subpsf_name, axis=1)

In [ ]:
# # 5. 정렬용 clean 컬럼 생성
# mapping_df["PSF_name_clean"] = mapping_df["PSF_name"].apply(clean_text)
# mapping_df["Sub-PSF_clean"] = mapping_df["Sub-PSF_fixed"].apply(clean_text)


# # 6. 순서표 merge
# mapping_df = mapping_df.merge(
#     order_df,
#     on=["PSF_name_clean", "Sub-PSF_clean"],
#     how="left"
# )


# # 7. PSF / Sub-PSF / level 순으로 정렬
# mapping_df = (
#     mapping_df
#     .sort_values(["psf_order", "subpsf_order", "level"])
#     .reset_index(drop=True)
# )

In [ ]:
# # 8. 매핑 안 된 항목 확인
# unmatched = (
#     mapping_df[
#         mapping_df["subpsf_order"].isna()
#     ][["PSF_name", "Sub-PSF"]]
#     .drop_duplicates()
# )

# unmatched

In [ ]:
# mapping_df.to_excel(
#     "subpsf_level_mapping_review.xlsx",
#     index=False
# )

## 2.2 mapping 수정한 데이터를 통해 PSF별 answer 테이블 생성

### 2.2.1 String 범주 형태로 저장

In [ ]:
answer_latest_file = max(
    Path("processed_data").glob("answer_*.csv"),
    key=lambda x: x.stat().st_mtime
)

answer_df = pd.read_csv(answer_latest_file)
answer_df.head(1)

mapping_df = pd.read_excel("processed_data/subpsf_level_mapping.xlsx")
answer_df.head(1)

In [ ]:
mapping_df.head(1)

In [ ]:
answer_df["conditions"] = answer_df["conditions"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [ ]:
# PSF별 테이블 저장용 dict
psf_tables = {}

In [ ]:
base_cols = [
    "empid",
    "survey_year",
    "survey_id",
    "set_id",
    "repeat_no",
    "PSF_name",
    "answer",
    "quality_100",
    "quality_95",
    "quality_90"
]


In [ ]:
if "repeat_no" not in answer_df.columns:
    answer_df = answer_df.copy()
    answer_df["repeat_no"] = (
        answer_df
        .groupby(["survey_id", "set_id", "PSF_name"])
        .cumcount()
    )

for psf, temp in answer_df.groupby("PSF_name"):
    temp = temp.copy().reset_index(drop=True)

    condition_wide = pd.DataFrame(
        temp["conditions"].tolist()
    )

    psf_table = pd.concat(
        [
            temp[base_cols].reset_index(drop=True),
            condition_wide.reset_index(drop=True)
        ],
        axis=1
    )

    psf_tables[psf] = psf_table


In [ ]:
for psf, table in psf_tables.items():
    print(psf, table.shape)
    display(table.head(1))

In [ ]:
import os

save_dir = "processed_data/psf_tables/string"
os.makedirs(save_dir, exist_ok=True)

for psf, table in psf_tables.items():

    file_name = (
        psf.replace("/", "_")
           .replace(" ", "_")
    )

    table.to_csv(
        f"{save_dir}/{file_name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

### 2.2.2 숫자 범주 형태로 저장

In [ ]:
for psf, table in psf_tables.items():
    psf_map = mapping_df[mapping_df["PSF_name"] == psf]

    for subpsf in psf_map["Sub-PSF"].unique():
        level_map = (
            psf_map[psf_map["Sub-PSF"] == subpsf]
            .set_index("Sub-PSF LV.")["level"]
            .to_dict()
        )

        if subpsf in table.columns:
            table[subpsf] = table[subpsf].map(level_map)

    psf_tables[psf] = table

In [ ]:
for psf, table in psf_tables.items():
    print(psf, table.shape)
    display(table.head(1))

In [ ]:
for psf, table in psf_tables.items():
    subpsf_cols = mapping_df.loc[
        mapping_df["PSF_name"] == psf,
        "Sub-PSF"
    ].unique()

    missing = table[list(subpsf_cols)].isna().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print("\n", psf)
        print(missing)

In [ ]:
import os

save_dir = "processed_data/psf_tables/number"
os.makedirs(save_dir, exist_ok=True)

for psf, table in psf_tables.items():

    file_name = (
        psf.replace("/", "_")
           .replace(" ", "_")
    )

    table.to_csv(
        f"{save_dir}/{file_name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

# 3. 응답 데이터 특성 분석

In [ ]:
import pandas as pd
import ast
import numpy as np
from pathlib import Path

pd.set_option('display.max_rows', None)      # 모든 행 출력
pd.set_option('display.max_columns', None)   # 모든 열 출력
pd.set_option('display.width', None)         # 출력 너비 제한 해제
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 길이 제한 해제

In [ ]:
answer_latest_file = max(
    Path("processed_data").glob("answer_*.csv"),
    key=lambda x: x.stat().st_mtime
)

answer_df = pd.read_csv(answer_latest_file)

job_latest_file = max(
    Path("processed_data").glob("job_*.csv"),
    key=lambda x: x.stat().st_mtime
)

job_df = pd.read_csv(job_latest_file)

health_latest_file = max(
    Path("processed_data").glob("health_*.csv"),
    key=lambda x: x.stat().st_mtime
)

health_df = pd.read_csv(health_latest_file)

In [ ]:
answer_df.head(1)

## 3.1 응답 분포 분석

### 3.1.1 년도별, 응답별, PSF 응답 분포 확인

In [ ]:
# 년도별 설문 수
answer_df["survey_year"].value_counts()

In [ ]:
# 년도별, 응답별 psf 설문 수
table = pd.crosstab(
    index=answer_df["PSF_name"],
    columns=[answer_df["survey_year"], answer_df["answer"]]
)

# PSF_name ↔ PSF 매핑
psf_order = (
    answer_df[["PSF_name", "PSF"]]
    .drop_duplicates()
    .set_index("PSF_name")["PSF"]
)

# PSF 숫자 기준 정렬
table = table.loc[
    psf_order.sort_values().index
]

display(table)

In [ ]:
# 년도별 psf 설문 수
table = pd.crosstab(
    index=answer_df["PSF_name"],
    columns=[answer_df["survey_year"]],
    margins=True,
    margins_name="합계"
)

# PSF_name ↔ PSF 매핑
psf_order = (
    answer_df[["PSF_name", "PSF"]]
    .drop_duplicates()
    .set_index("PSF_name")["PSF"]
)

# PSF 숫자 기준 정렬
table = table.loc[
    psf_order.sort_values().index
]

display(table)

In [ ]:
# 응답별 년도별 설문 수
table = pd.crosstab(
    index=answer_df["answer"],
    columns=[answer_df["survey_year"]],
    margins=True,
    margins_name="합계"
)

display(table)

### 3.1.2 인원별 응답 분석

In [ ]:
answer_df.groupby("survey_year")["empid"].nunique()

In [ ]:
answer_df["empid"].nunique()

### 3.1.3 응답자 특성 분석

In [ ]:
job_df.head(1)

In [ ]:
# 직무 분포
print(job_df["job"].value_counts())

In [ ]:
# 경력 분포
job_df["career_group"] = pd.cut(
    job_df["years"],
    bins=[-1, 4, 9, 19, 100],
    labels=["5년 미만", "5~10년", "10~20년", "20년 이상"]
)

career_table = (
    job_df["career_group"]
    .value_counts(sort=False)
    .rename_axis("경력")
    .reset_index(name="응답자 수")
)

career_table["비율(%)"] = (
    career_table["응답자 수"] / len(job_df) * 100
).round(1)

print(career_table)

In [ ]:
job_df["work_type_clean"] = job_df["work_type_clean"].replace({
    "주주야비휴휴": "기타",
    "통상일긍": "기타",
    "통상일근": "기타",
    "야간격일O조": "기타",
    "야간격일A조": "기타",
    "시차근무": "기타"
})

work_table = (
    job_df["work_type_clean"]
    .value_counts()
    .rename_axis("근무형태")
    .reset_index(name="응답자 수")
)

work_table["비율(%)"] = (
    work_table["응답자 수"] / len(job_df) * 100
).round(1)

print(work_table)

In [ ]:
health_df.head(1)

In [ ]:
# 연령대 분포
health_df["age_group"].value_counts()

## 3.2 작업절차 PSF 확인

In [ ]:
psf_string_dir = Path("processed_data/psf_tables/string")
psf_string_files = sorted(psf_string_dir.glob("*.csv"))

if not psf_string_files:
    raise FileNotFoundError(f"PSF string table CSV를 찾을 수 없습니다: {psf_string_dir}")

psf_string_tables = {
    file.stem.replace("_", " "): pd.read_csv(file)
    for file in psf_string_files
}

In [ ]:
PSF_작업절차 = psf_string_tables["작업절차"]
PSF_작업절차.shape

In [ ]:
PSF_작업절차.head(1)

In [ ]:
env_cols = [
    "역할 분담 명료화", "작업 계획 적절성", "작업절차 및 규정", "작업자간 협업 정도", "작업 관리 및 감독 적절성"
]

for col in env_cols:
    print(f"\n[{col}]")
    print(PSF_작업절차[col].value_counts(dropna=False))

In [ ]:
answer_order = ["나쁨", "보통", "좋음"]

for col in env_cols:
    print(f"\n{'='*50}")
    print(col)

    tab = pd.crosstab(
        PSF_작업절차[col],
        PSF_작업절차["answer"],
        normalize="index"
    )

    tab = tab.reindex(columns=[c for c in answer_order if c in tab.columns])

    display(tab.round(3))

## 3.3 PSF별 불량 응답 확인

In [ ]:
# psf_tables/string, job, health 데이터 불러오기
psf_string_dir = Path("processed_data/psf_tables/string")
psf_string_files = sorted(psf_string_dir.glob("*.csv"))

if not psf_string_files:
    raise FileNotFoundError(f"PSF string table CSV를 찾을 수 없습니다: {psf_string_dir}")

psf_string_tables = {
    file.stem.replace("_", " "): pd.read_csv(file)
    for file in psf_string_files
}

psf_string_tables["교육 및 훈련"].head(1)

In [ ]:
answer_latest_file = max(
    Path("processed_data").glob("answer_*_new_quality.csv"),
    key=lambda x: x.stat().st_mtime
)

answer_df = pd.read_csv(answer_latest_file)
answer_df.head(1)

In [ ]:
for name, df in psf_string_tables.items():
    print(name)
    print(df["quality_95"].value_counts(dropna=False))
    print(df["quality_100"].value_counts(dropna=False))
    print(df["quality_90"].value_counts(dropna=False))
    print()

In [ ]:
bad95 = pd.concat([
    df.loc[df["quality_95"] == "불성실", "empid"]
    for df in psf_string_tables.values()
]).drop_duplicates()

bad100 = pd.concat([
    df.loc[df["quality_100"] == "불성실", "empid"]
    for df in psf_string_tables.values()
]).drop_duplicates()

print(len(bad95), len(bad100))

print("100에는 있는데 95에는 없는 사람:",
      len(set(bad100) - set(bad95)))

print("95에는 있는데 100에는 없는 사람:",
      len(set(bad95) - set(bad100)))

In [ ]:
# 100% 단일 응답 기준 불성실 비율
for name, df in psf_string_tables.items():
    print(name)
    print(df["quality_100"].value_counts())
    # 추가: 고유 응답자 수
    bad_empids = df.loc[df["quality_100"] == "불성실", "empid"].nunique()
    print(f"고유 불성실 응답자 수: {bad_empids}")
    print()

bad_empids_100 = pd.concat(
    [
        df.loc[df["quality_100"] == "불성실", "empid"]
        for df in psf_string_tables.values()
    ],
    ignore_index=True
)

print(f"100% 기준 불성실 고유 응답자 수: {bad_empids_100.nunique()}")

In [ ]:
# 95% 단일 응답 기준 불성실 응답자(고유 empid) 수
for name, df in psf_string_tables.items():
    print(name)

    # 기존: 응답 수
    print(df["quality_95"].value_counts())

    # 추가: 고유 응답자 수
    bad_empids = df.loc[df["quality_95"] == "불성실", "empid"].nunique()
    print(f"고유 불성실 응답자 수: {bad_empids}")

    print()

bad_empids_100 = pd.concat(
    [
        df.loc[df["quality_95"] == "불성실", "empid"]
        for df in psf_string_tables.values()
    ],
    ignore_index=True
)

print(f"95% 기준 불성실 고유 응답자 수: {bad_empids_100.nunique()}")

In [ ]:
# 95% 단일 응답 기준 불성실 비율
for name, df in psf_string_tables.items():
    print(name)
    print(df["quality_90"].value_counts())
    print()


all_psf = pd.concat(psf_string_tables.values(), ignore_index=True)

target_ids = (
    all_psf.loc[
        (all_psf["quality_100"] == "불성실") &
        (all_psf["quality_95"] == "성실"),
        "empid"
    ]
    .unique()
)

print(f"대상 empid 수: {len(target_ids)}")
print(target_ids)


In [ ]:
(
    answer_df[
        answer_df["empid"].isin(target_ids)
    ][
        [
            "empid",
            "survey_year",
            "survey_id",
            "PSF",
            "PSF_name",
            "quality_100",
            "quality_95",
            "answer"
        ]
    ]
    .groupby(
        [
            "empid",
            "survey_year",
            "survey_id",
            "PSF_name",
            "PSF",
            "quality_100",
            "quality_95"
        ]
    )["answer"]
    .value_counts()
    .reset_index(name="count")
)

# 4. 환경 데이터 전처리

## 4.1 환경 데이터 전처리 1 (전체 변수 기반)
- 현재 낙뢰랑 안개는 모두 발생하는 것으로 되어있음.
- 환경에서 나쁨과 좋음의 비율은 약 7:3으로 낙뢰랑 안개가 큰 원인으로 파악됨
- 이에 다른 요소들의 영향력을 먼저 계산한 뒤 이것을 바탕으로 낙뢰와 안개의 영향력을 계산하는 방향으로

### 4.1.1 환경 요인별 영향도 분석

In [ ]:
# psf_tables/string, job, health 데이터 불러오기
psf_string_dir = Path("processed_data/psf_tables/string")
psf_string_files = sorted(psf_string_dir.glob("*.csv"))

if not psf_string_files:
    raise FileNotFoundError(f"PSF string table CSV를 찾을 수 없습니다: {psf_string_dir}")

psf_string_tables = {
    file.stem.replace("_", " "): pd.read_csv(file)
    for file in psf_string_files
}

In [ ]:
PSF_환경 = psf_string_tables["환경"]
PSF_환경.shape

In [ ]:
env_cols = [
    "비", "낙뢰", "소음", "안개", "통풍", "풍속", "황사",
    "마찰계수", "실내 습도", "실내 온도", "실외 습도",
    "실외 온도", "작업공간 크기", "업무공간 가시성"
]

for col in env_cols:
    print(f"\n[{col}]")
    print(PSF_환경[col].value_counts(dropna=False))

In [ ]:
answer_order = ["매우나쁨", "나쁨", "보통", "좋음", "매우좋음"]

for col in env_cols:
    print(f"\n{'='*50}")
    print(col)

    tab = pd.crosstab(
        PSF_환경[col],
        PSF_환경["answer"],
        normalize="index"
    )

    tab = tab.reindex(columns=[c for c in answer_order if c in tab.columns])

    display(tab.round(3))

In [ ]:
# 평균 응답 확인
base = (
    PSF_환경["answer"]
    .value_counts(normalize=True)
    .sort_index()
)

base

In [ ]:
pd.crosstab(
    PSF_환경["업무공간 가시성"],
    PSF_환경["answer"],
    normalize="index"
).round(3)

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np

def cramers_v(x, y):

    table = pd.crosstab(x, y)

    if min(table.shape) <= 1:
        return np.nan

    chi2 = chi2_contingency(table)[0]
    n = table.sum().sum()

    return np.sqrt(
        chi2 /
        (n * (min(table.shape)-1))
    )

#### 4.2.1.1 낙뢰와 안개를 제외한 변수별 영향력 계산

In [ ]:
env_cols = [
    "비",
    "소음",
    "통풍",
    "풍속",
    "황사",
    "마찰계수",
    "실내 습도",
    "실내 온도",
    "실외 습도",
    "실외 온도",
    "작업공간 크기",
    "업무공간 가시성"
]

result = []

for col in env_cols:

    score = cramers_v(
        PSF_환경[col],
        PSF_환경["answer"]
    )

    result.append([col, score])

result = (
    pd.DataFrame(
        result,
        columns=["변수", "영향력"]
    )
    .sort_values("영향력", ascending=False)
)

result

In [ ]:
env_cols = [
    "비",
    "소음",
    "통풍",
    "풍속",
    "황사",
    "마찰계수",
    "실내 습도",
    "실내 온도",
    "실외 습도",
    "실외 온도",
    "작업공간 크기",
    "업무공간 가시성"
]

mca_df = PSF_환경[env_cols + ["answer"]].astype(str)

In [ ]:
import prince

mca = prince.MCA(
    n_components=2,
    random_state=42
)

mca = mca.fit(mca_df)

In [ ]:
coords = mca.column_coordinates(mca_df)

coords.head()

In [ ]:
coords.sort_values(0)

In [ ]:
mca.eigenvalues_

In [ ]:
mca.percentage_of_variance_

In [ ]:
coords["dist"] = np.sqrt(coords[0]**2 + coords[1]**2)
coords.sort_values("dist", ascending=False)

### 4.1.2 낙뢰, 안개 제외한 변수 중요도 종합표 생성

#### 4.1.2.1 변수별 MCA 중요도 추출

In [ ]:
import numpy as np

coords2 = coords.copy()

coords2["mca_dist"] = np.sqrt(
    coords2[0]**2 +
    coords2[1]**2
)

coords2 = coords2.sort_values(
    "mca_dist",
    ascending=False
)

display(coords2)

In [ ]:
coords2["변수"] = (
    coords2.index
    .str.split("__")
    .str[0]
)

In [ ]:
mca_importance = (
    coords2
    .groupby("변수")["mca_dist"]
    .max()
    .reset_index()
)

mca_importance.sort_values("mca_dist",ascending=False)

#### 4.1.2.2 변수별 Cramers V 중요도 추출

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np

def cramers_v(x, y):

    table = pd.crosstab(x, y)

    if min(table.shape) <= 1:
        return np.nan

    chi2 = chi2_contingency(table)[0]
    n = table.sum().sum()

    return np.sqrt(
        chi2 / (n * (min(table.shape)-1))
    )

env_cols = [
    "비",
    "소음",
    "통풍",
    "풍속",
    "황사",
    "마찰계수",
    "실내 습도",
    "실내 온도",
    "실외 습도",
    "실외 온도",
    "작업공간 크기",
    "업무공간 가시성"
]

effect_df = []

for col in env_cols:

    effect_df.append([
        col,
        cramers_v(
            PSF_환경[col],
            PSF_환경["answer"]
        )
    ])

effect_df = (
    pd.DataFrame(
        effect_df,
        columns=["변수","answer_CramersV"]
    )
    .sort_values(
        "answer_CramersV",
        ascending=False
    )
)

display(effect_df)

#### 4.1.2.3 변수별 Catboost 중요도 추출

In [ ]:
from catboost import CatBoostClassifier

X = PSF_환경[env_cols].astype(str)
y = PSF_환경["answer"].astype(str)

model = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=500,
    depth=4,
    learning_rate=0.05,
    random_seed=42,
    verbose=False
)

model.fit(
    X,
    y,
    cat_features=list(range(len(env_cols)))
)

importance = pd.DataFrame({
    "변수": env_cols,
    "importance": model.get_feature_importance()
}).sort_values(
    "importance",
    ascending=False
)

display(importance)

#### 4.1.2.4 종합 중요도 추출

In [ ]:
# 중요도 병합
rank_df = (
    effect_df
    .merge(
        importance[["변수","importance"]],
        on="변수"
    )
    .merge(
        mca_importance,
        on="변수"
    )
)

display(rank_df)

In [ ]:
# 정규화 수행
for col in [
    "answer_CramersV",
    "importance",
    "mca_dist"
]:
    rank_df[col+"_norm"] = (
        rank_df[col] /
        rank_df[col].max()
    )

In [ ]:
# 종합 점수 추출
rank_df["total_score"] = (
    rank_df["answer_CramersV_norm"]
    + rank_df["importance_norm"]
    + rank_df["mca_dist_norm"]
) / 3

rank_df = rank_df.sort_values(
    "total_score",
    ascending=False
)

display(rank_df)

### 4.1.3 변수 분류 및 병합 수행

In [ ]:
perceived_cols = [
    "소음",
    "업무공간 가시성",
    "마찰계수",
    "실내 온도",
    "실내 습도",
    "통풍",
    "answer"
]

mca_df = PSF_환경[perceived_cols].astype(str)

In [ ]:
mca = prince.MCA(
    n_components=2,
    random_state=42
)

mca = mca.fit(mca_df)

In [ ]:
coords = mca.column_coordinates(mca_df)

display(coords)

In [ ]:
import numpy as np

coords["mca_dist"] = np.sqrt(
    coords[0]**2 +
    coords[1]**2
)

display(
    coords.sort_values(
        "mca_dist",
        ascending=False
    )
)

In [ ]:
coords["변수"] = (
    coords.index
    .str.split("__")
    .str[0]
)

coords["상태"] = (
    coords.index
    .str.split("__")
    .str[1]
)

display(
    coords.sort_values(
        [0, 1]
    )
)

## 4.2 환경 데이터 전처리 2 (Composite State 도출)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# psf_tables/string, job, health 데이터 불러오기
psf_string_dir = Path("processed_data/psf_tables/number")
psf_string_files = sorted(psf_string_dir.glob("*.csv"))

if not psf_string_files:
    raise FileNotFoundError(f"PSF string table CSV를 찾을 수 없습니다: {psf_string_dir}")

psf_string_tables = {
    file.stem.replace("_", " "): pd.read_csv(file)
    for file in psf_string_files
}

In [ ]:
PSF_환경 = psf_string_tables["환경"]
PSF_환경.shape

In [ ]:
env_df = PSF_환경.copy()
env_df.columns = env_df.columns.str.strip()

rename_map = {
    "실내 습도": "실내습도",
    "실내 온도": "실내온도",
    "실외 습도": "실외습도",
    "실외 온도": "실외온도",
}
env_df = env_df.rename(columns=rename_map)

perceived_env_groups = {
    "Thermal_Comfort": ["실내온도", "실내습도", "통풍"],
    "Visual_Environment": ["업무공간 가시성", "안개", "황사"],
    "Surface_Safety": ["마찰계수", "비"],
    "Workspace": ["작업공간 크기"],
    "Acoustic_Environment": ["소음"],
    "Outdoor_Climate": ["실외온도", "실외습도", "풍속", "낙뢰", "비"],
}

In [ ]:
def make_composite_state(df, group_cols, state_name, n_components=3, random_state=42):
    result = df.copy()
    X = result[group_cols].copy()
    X = X.apply(pd.to_numeric, errors="coerce")

    # 숫자형 변환
    X = X.apply(pd.to_numeric, errors="coerce")

    # 결측값은 변수별 최빈값으로 대체
    for col in group_cols:
        if X[col].isna().any():
            X[col] = X[col].fillna(X[col].mode().iloc[0])

    # 단일 변수 체감환경은 GMM 생략
    if len(group_cols) == 1:
        col = group_cols[0]

        single_var_map = {
            0: 0,
            1: 1,
            2: 2,
        }

        result[state_name] = X[col].map(single_var_map)
        result[state_name + "_score"] = X[col]
        result[state_name + "_method"] = "direct_mapping"

        summary = (
            X.groupby(col)
            .size()
            .reset_index(name="count")
            .rename(columns={col: "state_score"})
        )

        summary["label"] = summary["state_score"].map(single_var_map)

        return result, None, summary

    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="full",
        random_state=random_state,
        n_init=20
    )

    cluster = gmm.fit_predict(X)

    cluster_mean = pd.DataFrame(gmm.means_, columns=group_cols)
    cluster_mean["mean_score"] = cluster_mean.mean(axis=1)

    ordered_clusters = cluster_mean.sort_values("mean_score").index.tolist()

    label_map = {
        ordered_clusters[0]: 0,
        ordered_clusters[1]: 1,
        ordered_clusters[2]: 2,
    }

    result[state_name] = [label_map[c] for c in cluster]
    result[state_name + "_cluster"] = cluster

    return result, gmm, cluster_mean

In [ ]:
env_composite = env_df.copy()

gmm_models = {}
cluster_summary = {}

for state_name, cols in perceived_env_groups.items():
    env_composite, gmm, summary = make_composite_state(
        env_composite,
        group_cols=cols,
        state_name=state_name,
        n_components=3,
        random_state=42
    )

    gmm_models[state_name] = gmm
    cluster_summary[state_name] = summary

In [ ]:
composite_cols = list(perceived_env_groups.keys())

env_composite[composite_cols].head()

In [ ]:
for name, summary in cluster_summary.items():
    print(f"\n[{name}]")
    display(summary.sort_values("mean_score") if "mean_score" in summary.columns else summary)

In [ ]:
meta_cols = [
    c for c in ["empid", "survey_id", "survey_year", "PSF_name", "answer", "quality_100", "quality_95", "quality_90"]
    if c in env_composite.columns
]

env_model_df = env_composite[
    meta_cols + composite_cols
].copy()

env_model_df.head()

In [ ]:
print("기존 관측 환경 시나리오 수:", env_df[list(set(sum(perceived_env_groups.values(), [])))].drop_duplicates().shape[0])
print("재구성 후 관측 체감환경 시나리오 수:", env_model_df[composite_cols].drop_duplicates().shape[0])
print("이론상 최대 체감환경 시나리오 수:", 3 ** len(composite_cols))

In [ ]:
import os

save_dir = "processed_data/psf_tables/number/환경_new_category.csv"
env_model_df.to_csv(save_dir, index=False, encoding="utf-8-sig")

# 5. 나머지 전처리

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_rows', None)      # 모든 행 출력
pd.set_option('display.max_columns', None)   # 모든 열 출력
pd.set_option('display.width', None)         # 출력 너비 제한 해제
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 길이 제한 해제

In [ ]:
# 공통 컬럼 정리 함수
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace("\u00a0", " ", regex=False)
        .str.strip()
    )
    return df

In [ ]:
def make_composite_state(df, group_cols, state_name, n_components=3, random_state=42):
    result = df.copy()
    X = result[group_cols].copy()
    X = X.apply(pd.to_numeric, errors="coerce")

    # 숫자형 변환
    X = X.apply(pd.to_numeric, errors="coerce")

    # 결측값은 변수별 최빈값으로 대체
    for col in group_cols:
        if X[col].isna().any():
            X[col] = X[col].fillna(X[col].mode().iloc[0])

    # 단일 변수 체감환경은 GMM 생략
    if len(group_cols) == 1:
        col = group_cols[0]

        single_var_map = {
            0: 0,
            1: 1,
            2: 2,
        }

        result[state_name] = X[col].map(single_var_map)
        result[state_name + "_score"] = X[col]
        result[state_name + "_method"] = "direct_mapping"

        summary = (
            X.groupby(col)
            .size()
            .reset_index(name="count")
            .rename(columns={col: "state_score"})
        )

        summary["label"] = summary["state_score"].map(single_var_map)

        return result, None, summary

    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="full",
        random_state=random_state,
        n_init=20
    )

    cluster = gmm.fit_predict(X)

    cluster_mean = pd.DataFrame(gmm.means_, columns=group_cols)
    cluster_mean["mean_score"] = cluster_mean.mean(axis=1)

    ordered_clusters = cluster_mean.sort_values("mean_score").index.tolist()

    label_map = {
        ordered_clusters[0]: 0,
        ordered_clusters[1]: 1,
        ordered_clusters[2]: 2,
    }

    result[state_name] = [label_map[c] for c in cluster]
    result[state_name + "_cluster"] = cluster

    return result, gmm, cluster_mean

In [ ]:
from pathlib import Path
import joblib


GMM_MODEL_DIR = Path("models/gmm")
GMM_MODEL_DIR.mkdir(parents=True, exist_ok=True)


def save_gmm_bundle(
    gmm,
    group_cols,
    state_name,
    save_name,
    source_df=None,
):
    """
    학습된 GMM과 API 추론에 필요한 정보를 함께 저장한다.

    저장 내용
    ----------
    gmm
        학습된 GaussianMixture 모델

    label_map
        GMM cluster 번호 -> 0/1/2 상태 매핑

    group_cols
        해당 GMM 입력 변수와 순서

    state_name
        생성되는 composite feature 이름

    fill_values
        결측값 발생 시 사용할 학습 데이터 기준 최빈값
    """

    if gmm is None:
        print(f"[SKIP] {state_name}: 단일 변수이므로 GMM 저장 불필요")
        return

    # -----------------------------------------
    # cluster 번호 -> 좋음/보통/나쁨 0/1/2
    # -----------------------------------------
    cluster_mean = pd.DataFrame(
        gmm.means_,
        columns=group_cols,
    )

    cluster_mean["mean_score"] = (
        cluster_mean[group_cols]
        .mean(axis=1)
    )

    ordered_clusters = (
        cluster_mean
        .sort_values("mean_score")
        .index
        .tolist()
    )

    label_map = {
        int(cluster): int(label)
        for label, cluster
        in enumerate(ordered_clusters)
    }

    # -----------------------------------------
    # 학습 당시 결측값 대체값
    # -----------------------------------------
    fill_values = {}

    if source_df is not None:
        X = source_df[group_cols].copy()

        X = X.apply(
            pd.to_numeric,
            errors="coerce",
        )

        for col in group_cols:
            mode = X[col].mode(dropna=True)

            if not mode.empty:
                fill_values[col] = float(mode.iloc[0])

    # -----------------------------------------
    # API에서 필요한 것을 하나로 저장
    # -----------------------------------------
    bundle = {
        "gmm": gmm,
        "label_map": label_map,
        "group_cols": list(group_cols),
        "state_name": state_name,
        "fill_values": fill_values,
    }

    save_path = (
        GMM_MODEL_DIR
        / f"{save_name}.joblib"
    )

    joblib.dump(
        bundle,
        save_path,
    )

    print(f"[SAVE] {state_name}")
    print(f"       {save_path}")
    print(f"       group_cols = {group_cols}")
    print(f"       label_map = {label_map}")

## 5.1 장비 및 기기

In [ ]:
# psf_tables/string, job, health 데이터 불러오기
psf_string_dir = Path("processed_data/psf_tables/number")
psf_string_files = sorted(psf_string_dir.glob("*.csv"))

if not psf_string_files:
    raise FileNotFoundError(f"PSF string table CSV를 찾을 수 없습니다: {psf_string_dir}")

psf_string_tables = {
    file.stem.replace("_", " "): pd.read_csv(file)
    for file in psf_string_files
}

In [ ]:
PSF_장비 = psf_string_tables["장비 및 기기"]
PSF_장비.head(1)

In [ ]:
equip_df = clean_columns(PSF_장비)

equipment_groups = {
    "Equipment_Accessibility": ["장비 가용성(접근성)"],
    "Equipment_Quality": ["작업 도구의 적절성", "장비의 사용성"],
    "System_Support": ["컴퓨터 지원 시스템의 적절성", "타 기기와의 연계성"],
    "Documentation_Quality": ["장비 사용 매뉴얼의 질"],
    "Maintenance_Support": ["유지보수의 적절성"],
}

equip_composite = equip_df.copy()
equip_gmm_models = {}
equip_cluster_summary = {}

for state_name, cols in equipment_groups.items():
    equip_composite, gmm, summary = make_composite_state(
        equip_composite,
        group_cols=cols,
        state_name=state_name,
        n_components=3,
        random_state=42
    )

    equip_gmm_models[state_name] = gmm
    equip_cluster_summary[state_name] = summary

equip_composite_cols = list(equipment_groups.keys())

meta_cols = [
    c for c in [
        "empid", "survey_id", "survey_year", "PSF_name", "answer",
        "quality_100", "quality_95", "quality_90"
    ]
    if c in equip_composite.columns
]

equip_model_df = equip_composite[
    meta_cols + equip_composite_cols
].copy()

equip_model_df.head()

In [ ]:
save_gmm_bundle(
    gmm=equip_gmm,
    group_cols=thermal_cols,
    state_name="Thermal_Comfort",
    save_name="psf6_Thermal_Comfort",
    source_df=env_df,
)

In [ ]:
save_dir = "processed_data/psf_tables/number/장비_및_기기_new_category.csv"
equip_model_df.to_csv(save_dir, index=False, encoding="utf-8-sig")

## 5.2 교육 및 훈련

In [ ]:
# psf_tables/string, job, health 데이터 불러오기
psf_string_dir = Path("processed_data/psf_tables/number")
psf_string_files = sorted(psf_string_dir.glob("*.csv"))

if not psf_string_files:
    raise FileNotFoundError(f"PSF string table CSV를 찾을 수 없습니다: {psf_string_dir}")

psf_string_tables = {
    file.stem.replace("_", " "): pd.read_csv(file)
    for file in psf_string_files
}

In [ ]:
PSF_교육 = psf_string_tables["교육 및 훈련"]
PSF_교육.head(1)

In [ ]:
train_df = clean_columns(PSF_교육)

education_feature_cols = [
    "교육/훈련 프로그램 적절성",
    "교육 주기(빈도)",
    "상황에 대한 경험 정도(자주 또는 드물게 겪는지)",
    "직무능력 평가 빈도",
    "시스템 지식(원리적 지식)의 숙련도",
    "절차적 지식의 숙련도",
    "장비 사용 지식의 숙련도",
]


In [ ]:
training_groups = {
    "Training_Program": ["교육/훈련 프로그램 적절성"],
    "Training_Frequency": ["교육 주기(빈도)", "직무능력 평가 빈도"],
    "Experience_Level": ["상황에 대한 경험 정도(자주 또는 드물게 겪는지)"],
    "Technical_Knowledge": [
        "시스템 지식(원리적 지식)의 숙련도",
        "장비 사용 지식의 숙련도"
    ],
    "Procedural_Knowledge": ["절차적 지식의 숙련도"],
}

train_composite = train_df.copy()
train_gmm_models = {}
train_cluster_summary = {}

for state_name, cols in training_groups.items():
    train_composite, gmm, summary = make_composite_state(
        train_composite,
        group_cols=cols,
        state_name=state_name,
        n_components=3,
        random_state=42
    )

    train_gmm_models[state_name] = gmm
    train_cluster_summary[state_name] = summary

train_composite_cols = list(training_groups.keys())

meta_cols = [
    c for c in [
        "empid", "survey_id", "survey_year", "PSF_name", "answer",
        "quality_100", "quality_95", "quality_90"
    ]
    if c in train_composite.columns
]

train_model_df = train_composite[
    meta_cols + train_composite_cols
].copy()

train_model_df.head()

In [ ]:
save_dir = "processed_data/psf_tables/number/교육_및_훈련_new_category.csv"
train_model_df.to_csv(save_dir, index=False, encoding="utf-8-sig")

## 5.3 복잡성

In [ ]:
# psf_tables/string, job, health 데이터 불러오기
psf_string_dir = Path("processed_data/psf_tables/number")
psf_string_files = sorted(psf_string_dir.glob("*.csv"))

if not psf_string_files:
    raise FileNotFoundError(f"PSF string table CSV를 찾을 수 없습니다: {psf_string_dir}")

psf_string_tables = {
    file.stem.replace("_", " "): pd.read_csv(file)
    for file in psf_string_files
}

In [ ]:
PSF_복잡성 = psf_string_tables["복잡성"]
PSF_복잡성.head(1)

In [ ]:
complex_df = clean_columns(PSF_복잡성)

complexity_groups = {
    "Goal_Clarity": ["의사결정 목표의 명확성"],
    "Decision_Complexity": [
        "의사결정 대안의 수",
        "의사결정 판단 기준의 수"
    ],
    "Concurrent_Decision_Load": [
        "(동시에 처리 되어야 하는)의사결정 수"
    ],
    "Information_Complexity": [
        "관측되는 정보의 양",
        "(관측이 불가능하여)추론해야하는 정보의 양"
    ],
    "Information_Uncertainty": ["정보의 불확실성"],
}

complex_composite = complex_df.copy()
complex_gmm_models = {}
complex_cluster_summary = {}

for state_name, cols in complexity_groups.items():
    complex_composite, gmm, summary = make_composite_state(
        complex_composite,
        group_cols=cols,
        state_name=state_name,
        n_components=3,
        random_state=42
    )

    complex_gmm_models[state_name] = gmm
    complex_cluster_summary[state_name] = summary

complex_composite_cols = list(complexity_groups.keys())

meta_cols = [
    c for c in [
        "empid", "survey_id", "survey_year", "PSF_name", "answer",
        "quality_100", "quality_95", "quality_90"
    ]
    if c in complex_composite.columns
]

complex_model_df = complex_composite[
    meta_cols + complex_composite_cols
].copy()

complex_model_df.head()

In [ ]:
save_dir = "processed_data/psf_tables/number/복잡성_new_category.csv"
complex_model_df.to_csv(save_dir, index=False, encoding="utf-8-sig")

# X

In [ ]:
# psf_tables/string, job, health 데이터 불러오기
psf_string_dir = Path("processed_data/psf_tables/string")
psf_string_files = sorted(psf_string_dir.glob("*.csv"))

if not psf_string_files:
    raise FileNotFoundError(f"PSF string table CSV를 찾을 수 없습니다: {psf_string_dir}")

psf_string_tables = {
    file.stem.replace("_", " "): pd.read_csv(file)
    for file in psf_string_files
}

job_latest_file = max(
    Path("processed_data").glob("job_*.csv"),
    key=lambda x: x.stat().st_mtime
)

job_df = pd.read_csv(job_latest_file)

health_latest_file = max(
    Path("processed_data").glob("health_*.csv"),
    key=lambda x: x.stat().st_mtime
)

health_df = pd.read_csv(health_latest_file)

In [ ]:
psf_string_tables.keys()

In [ ]:
job_df.head(1)

In [ ]:
health_df.head(1)

## X.X answer, job 데이터 분석

In [ ]:
job_merge_cols = ["empid", "job", "years", "work_type_clean", "job_family", "grade"]
health_merge_cols = ["empid", "age", "gender", "age_group"]

profile_df = job_df[job_merge_cols].merge(
    health_df[health_merge_cols],
    on="empid",
    how="left"
)

merged_psf_tables = {
    psf: table.merge(
        profile_df,
        on="empid",
        how="left"
    )
    for psf, table in psf_string_tables.items()
}

In [ ]:
merged_psf_tables["작업부하"].head(1)

In [ ]:
# PSF별 매칭 안 된 사람 확인
merge_check = pd.DataFrame({
    psf: {
        "job_missing": table["job"].isna().sum(),
        "age_missing": table["age"].isna().sum(),
        "gender_missing": table["gender"].isna().sum()
    }
    for psf, table in merged_psf_tables.items()
}).T.sort_index()

merge_check

### X.X.1 작업부하 분석

In [ ]:
workload = merged_psf_tables["작업부하"].copy()
workload["age"] = pd.to_numeric(workload["age"], errors="coerce").astype("Int64")

In [ ]:
person_df = workload[
    ["empid", "age", "job", "age_group"]
].drop_duplicates("empid")

In [ ]:
# 작업부하 응답자 나이 분포
person_df["age_group"].value_counts().sort_index()

In [ ]:
# 작업부하 응답자 직업 분포
person_df["job"].value_counts()

In [ ]:
answer_order = ["매우나쁨", "나쁨", "보통", "좋음", "매우좋음"]

def answer_distribution(df, index_cols):
    if isinstance(index_cols, str):
        index_cols = [index_cols]

    index = [df[col] for col in index_cols]
    if len(index) == 1:
        index = index[0]

    count = pd.crosstab(
        index=index,
        columns=df["answer"],
        dropna=False
    )

    ordered_cols = [col for col in answer_order if col in count.columns]
    ordered_cols += [col for col in count.columns if col not in ordered_cols]
    count = count[ordered_cols]

    ratio = count.div(count.sum(axis=1), axis=0).round(4)

    return {
        "count": count,
        "ratio": ratio
    }

workload_answer_dist = {
    "job": answer_distribution(workload, "job"),
    "age": answer_distribution(workload, "age"),
    "job_age": answer_distribution(workload, ["job", "age_group"])
}

In [ ]:
# 직무별 응답 비율
display(workload_answer_dist["job"]["ratio"])

In [ ]:
workload[workload["quality_100"]=="불성실"].age_group.unique()

In [ ]:
# 직무별 응답 비율
display(workload_answer_dist["job_age"]["ratio"])

In [ ]:
work_time_dist = answer_distribution(
    workload,
    "근무시간"
)

work_time_dist["count"]

In [ ]:
target = workload[
    (workload["근무시간"] == "8시간 미만 근무할 경우") &
    (workload["answer"] == "매우나쁨")
]

target[
    [
        "empid",
        "수면시간",
        "작업시간",
        "근무시간대",
        "작업 허용 시간"
    ]
]

In [ ]:
pd.crosstab(
    workload["수면시간"],
    workload["answer"],
    normalize="index"
).round(3)

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np

def cramers_v(x, y):
    confusion = pd.crosstab(x, y)

    if confusion.shape[0] < 2 or confusion.shape[1] < 2:
        return np.nan

    chi2 = chi2_contingency(confusion)[0]
    n = confusion.sum().sum()

    return np.sqrt(
        chi2 / (
            n *
            min(
                confusion.shape[0]-1,
                confusion.shape[1]-1
            )
        )
    )

In [ ]:
psf_cols = [
    "근무시간",
    "수면시간",
    "작업시간",
    "근무시간대",
    "작업 허용 시간"
]

In [ ]:
results = []

for age in workload["age_group"].dropna().unique():

    temp = workload[
        workload["age_group"] == age
    ]

    for psf in psf_cols:

        results.append({
            "age_group": age,
            "psf": psf,
            "cramers_v": cramers_v(
                temp[psf],
                temp["answer"]
            )
        })

result_df = pd.DataFrame(results)

In [ ]:
result_df.sort_values(
    ["age_group", "cramers_v"],
    ascending=[True, False]
)